In [8]:
import pandas as pd

# Load the raw CSV
raw_path = "data/raw/sephora_products.csv"
df = pd.read_csv(raw_path)

print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("Column Names:", df.columns.tolist())

Total Rows: 9168
Total Columns: 21
Column Names: ['id', 'brand', 'category', 'name', 'size', 'rating', 'number_of_reviews', 'love', 'price', 'value_price', 'URL', 'MarketingFlags', 'MarketingFlags_content', 'options', 'details', 'how_to_use', 'ingredients', 'online_only', 'exclusive', 'limited_edition', 'limited_time_offer']


In [9]:
# 1. Check for missing values in critical columns
print("Missing values per column:")
print(df.isnull().sum())

# 2. View 5 sample rows across the main target columns
df[["brand", "name", "category", "ingredients", "price"]].head(5)

Missing values per column:
id                        0
brand                     0
category                  0
name                      0
size                      0
rating                    0
number_of_reviews         0
love                      0
price                     0
value_price               0
URL                       0
MarketingFlags            0
MarketingFlags_content    0
options                   0
details                   0
how_to_use                0
ingredients               0
online_only               0
exclusive                 0
limited_edition           0
limited_time_offer        0
dtype: int64


,brand,name,category,ingredients,price
0,Acqua Di Parma,Blu Mediterraneo MINIATURE Set,Fragrance,Arancia di Capri Eau de Toilette: Alcohol Dena...,66.0
1,Acqua Di Parma,Colonia,Cologne,unknown,66.0
2,Acqua Di Parma,Arancia di Capri,Perfume,Alcohol Denat.- Water- Fragrance- Limonene- Li...,180.0
3,Acqua Di Parma,Mirto di Panarea,Perfume,unknown,120.0
4,Acqua Di Parma,Colonia Miniature Set,Fragrance,Colonia: Alcohol Denat.- Water- Fragrance- Lim...,72.0


In [10]:
# List all unique category names and their counts
category_counts = df["category"].value_counts()
print(f"Total Unique Categories: {len(category_counts)}\n")
print(category_counts)

Total Unique Categories: 143

category
Perfume                      665
Moisturizers                 451
Face Serums                  384
Value & Gift Sets            378
Face Wash & Cleansers        247
                            ... 
Wellness                       1
High Tech Tools                1
Hair Styling & Treatments      1
Curls & Coils                  1
Lid Shadow Brush               1
Name: count, Length: 143, dtype: int64


In [11]:
# Inspect 5 random non-null ingredient entries
sample_ingredients = df["ingredients"].dropna().sample(5, random_state=42).tolist()

for i, ing_text in enumerate(sample_ingredients, 1):
    print(f"--- Sample {i} ---")
    print(ing_text[:300]) # Prints first 300 characters
    print("\n")

--- Sample 1 ---
Sucrose- Morrocan Lava Clay- Ammonium Lauryl Sulfate- Water- Butylene Glycol Cocamide Mea- Peg-100 Sterate- Hydrated Silica- Glyceryl Caprylate/Caprate- Plankton Extract- Fragrance- Peg-2 Dimeadowfoamamidoethylmonium Methosulfate- Pentylene Glycol- Hexylene Glycol- Phenoxyethanol- Linalool- Limonene


--- Sample 2 ---
Algae (Seaweed) Extract- Cyclopentasiloxane- Petrolatum- Glyceryl Distearate- Phenyl Trimethicone- Butylene Glycol- Hydrogenated Vegetable Oil- Cholesterol- Butyrospermum Parkii (Shea Butter)- Steareth-10- Dimethicone- Glyceryl Stearate Se- Polysilicone-11- Sesamum Indicum (Sesame) Seed Oil- Medicag


--- Sample 3 ---
unknown


--- Sample 4 ---
 -Squalane: Helps restore skin’s natural moisture balance to keep it extra soft and supple.   
-Octinoxate: Helps prevent skin from burning in the sun.
 Avobenzone 3%- Homosalate 7%- Octisalate 5%- and Octocrylene 5%: Provide sun protection. 
Water-
Propylene Glycol-
Dicaprylyl Ether-
Glycerin-
Peg-


--- Sample 5 

In [12]:
# 1. Inspect all categories to pick exact skincare matches
print("--- All Categories Containing 'Face', 'Skin', 'Eye', 'Moisturizer', 'Serum', 'Clean' ---")
skincare_matches = df[df['category'].str.contains(r'Face|Skin|Eye|Moisturizer|Serum|Clean|Sunscreen|Mask|Toner|Peel', case=False, na=False)]['category'].value_counts()
print(skincare_matches)

--- All Categories Containing 'Face', 'Skin', 'Eye', 'Moisturizer', 'Serum', 'Clean' ---
category
Moisturizers                451
Face Serums                 384
Face Wash & Cleansers       247
Face Masks                  230
Eye Palettes                202
Eye Creams & Treatments     191
Face Brushes                183
Face Primer                 144
Eyeliner                    126
Eyebrow                     107
Eye Brushes                 100
Toners                       87
Face Oils                    84
Hair Masks                   80
Eyeshadow                    78
Face Sunscreen               76
Sheet Masks                  56
Facial Peels                 55
False Eyelashes              52
Skincare                     48
Facial Cleansing Brushes     44
Face Sets                    44
Moisturizer & Treatments     31
Eye Primer                   30
Eye Sets                     29
Brush Cleaners               26
Face Wipes                   22
Eye Masks                    21
For Fa

# Filter out makeup and tools and retain only active skincare categories

In [14]:
# 1. Define exact skincare categories to keep
target_categories = [
    "Moisturizers",
    "Face Serums",
    "Face Wash & Cleansers",
    "Face Wash",
    "Toners",
    "Facial Peels",
    "Face Oils",
    "Face Sunscreen",
    "Sunscreen",
    "Eye Creams & Treatments",
    "Eye Cream",
    "Moisturizer & Treatments"
]

# 2. Filter dataframe
df_clean = df[df["category"].isin(target_categories)].copy()

# 3. Drop missing and 'unknown' ingredient entries
df_clean = df_clean[df_clean["ingredients"].str.lower().str.strip() != "unknown"]
df_clean = df_clean.dropna(subset=["ingredients", "name", "brand"])

# 4. Remove duplicate products
df_clean = df_clean.drop_duplicates(subset=["name", "brand"])

print(f"Cleaned Skincare Products Remaining: {len(df_clean)}")

Cleaned Skincare Products Remaining: 1599


In [15]:
def map_standard_category(cat):
    cat_lower = str(cat).lower()
    if "clean" in cat_lower or "wash" in cat_lower:
        return "Cleanser"
    elif "toner" in cat_lower or "peel" in cat_lower:
        return "Toner & Exfoliant"
    elif "serum" in cat_lower or "oil" in cat_lower:
        return "Serum"
    elif "sunscreen" in cat_lower:
        return "Sunscreen"
    elif "eye" in cat_lower:
        return "Eye Cream"
    else:
        return "Moisturizer"

df_clean["standard_category"] = df_clean["category"].apply(map_standard_category)
print("Standardized Category Distribution:")
print(df_clean["standard_category"].value_counts())

Standardized Category Distribution:
standard_category
Moisturizer          474
Serum                458
Cleanser             259
Eye Cream            197
Toner & Exfoliant    133
Sunscreen             78
Name: count, dtype: int64


In [16]:
import os

# 1. Keep only essential columns
columns_to_keep = ["brand", "name", "standard_category", "ingredients", "price", "rating"]
df_final = df_clean[columns_to_keep].reset_index(drop=True)

# 2. Ensure directory exists
os.makedirs("data/processed", exist_ok=True)

# 3. Save to processed folder
output_path = "data/processed/products_clean.csv"
df_final.to_csv(output_path, index=False)

print(f"Successfully saved {len(df_final)} rows to {output_path}")
df_final.head(5)

Successfully saved 1599 rows to data/processed/products_clean.csv


,brand,name,standard_category,ingredients,price,rating
0,Algenist,GENIUS Liquid Collagen,Serum,-Patented Alguronic Acid: Naturally sourced a...,115.0,4.0
1,Algenist,GENIUS Sleeping Collagen,Moisturizer,-Patented Alguronic Acid: Naturally sourced a...,98.0,4.5
2,Algenist,GENIUS Ultimate Anti-Aging Cream,Moisturizer,-Patented Alguronic Acid: Naturally sourced a...,112.0,4.5
3,Algenist,Complete Eye Renewal Balm,Eye Cream,-Patented Alguronic Acid: Visibly minimizes t...,68.0,4.0
4,Algenist,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,Sunscreen,-Patented Alguronic Acid: Helps minimize the ...,28.0,4.5


# notebook to generate the two knowledge-base tables

In [17]:
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

concern_data = [
    # Acne & Blemishes
    {"concern": "Acne & Blemishes", "ingredient": "salicylic acid", "weight": 1.0},
    {"concern": "Acne & Blemishes", "ingredient": "benzoyl peroxide", "weight": 1.0},
    {"concern": "Acne & Blemishes", "ingredient": "azelaic acid", "weight": 0.9},
    {"concern": "Acne & Blemishes", "ingredient": "niacinamide", "weight": 0.8},
    {"concern": "Acne & Blemishes", "ingredient": "zinc", "weight": 0.8},
    {"concern": "Acne & Blemishes", "ingredient": "tea tree", "weight": 0.7},

    # Hyperpigmentation & Dark Spots
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "vitamin c", "weight": 1.0},
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "ascorbic acid", "weight": 1.0},
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "alpha arbutin", "weight": 0.9},
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "tranexamic acid", "weight": 0.9},
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "azelaic acid", "weight": 0.9},
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "niacinamide", "weight": 0.8},
    {"concern": "Hyperpigmentation & Dark Spots", "ingredient": "glycolic acid", "weight": 0.8},

    # Anti-Aging & Fine Lines
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "retinol", "weight": 1.0},
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "retinal", "weight": 1.0},
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "tretinoin", "weight": 1.0},
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "bakuchiol", "weight": 0.8},
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "peptide", "weight": 0.9},
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "collagen", "weight": 0.7},
    {"concern": "Anti-Aging & Fine Lines", "ingredient": "adenosine", "weight": 0.7},

    # Dryness & Barrier Repair
    {"concern": "Dryness & Barrier Repair", "ingredient": "hyaluronic acid", "weight": 1.0},
    {"concern": "Dryness & Barrier Repair", "ingredient": "sodium hyaluronate", "weight": 1.0},
    {"concern": "Dryness & Barrier Repair", "ingredient": "ceramide", "weight": 1.0},
    {"concern": "Dryness & Barrier Repair", "ingredient": "squalane", "weight": 0.9},
    {"concern": "Dryness & Barrier Repair", "ingredient": "glycerin", "weight": 0.9},
    {"concern": "Dryness & Barrier Repair", "ingredient": "panthenol", "weight": 0.8},

    # Redness & Sensitivity
    {"concern": "Redness & Sensitivity", "ingredient": "centella asiatica", "weight": 1.0},
    {"concern": "Redness & Sensitivity", "ingredient": "cica", "weight": 1.0},
    {"concern": "Redness & Sensitivity", "ingredient": "madecassoside", "weight": 1.0},
    {"concern": "Redness & Sensitivity", "ingredient": "allantoin", "weight": 0.8},
    {"concern": "Redness & Sensitivity", "ingredient": "azelaic acid", "weight": 0.9},
    {"concern": "Redness & Sensitivity", "ingredient": "panthenol", "weight": 0.8},

    # Dullness & Texture
    {"concern": "Dullness & Texture", "ingredient": "glycolic acid", "weight": 1.0},
    {"concern": "Dullness & Texture", "ingredient": "lactic acid", "weight": 0.9},
    {"concern": "Dullness & Texture", "ingredient": "mandelic acid", "weight": 0.8},
    {"concern": "Dullness & Texture", "ingredient": "gluconolactone", "weight": 0.8},
    {"concern": "Dullness & Texture", "ingredient": "vitamin c", "weight": 0.9}
]

df_concern_map = pd.DataFrame(concern_data)
df_concern_map.to_csv("data/concern_ingredient_map.csv", index=False)
print(f"Saved concern_ingredient_map.csv with {len(df_concern_map)} mappings.")

Saved concern_ingredient_map.csv with 37 mappings.


In [18]:
conflict_data = [
    {
        "ingredient_a": "retinol",
        "ingredient_b": "glycolic acid",
        "severity": "High",
        "conflict_type": "Chemical Contraindication",
        "reason": "Simultaneous application of retinoids and potent AHAs causes acute epidermal barrier degradation, peeling, and erythema.",
        "solution": "Split routines: use Glycolic Acid in AM or alternate PM nights."
    },
    {
        "ingredient_a": "retinol",
        "ingredient_b": "salicylic acid",
        "severity": "High",
        "conflict_type": "Chemical Contraindication",
        "reason": "Combining BHA exfoliants with retinoids excessively strips lipid bilayers, leading to severe barrier damage.",
        "solution": "Use Salicylic Acid in AM and Retinol in PM."
    },
    {
        "ingredient_a": "retinol",
        "ingredient_b": "benzoyl peroxide",
        "severity": "High",
        "conflict_type": "Active Deactivation",
        "reason": "Benzoyl peroxide oxidizes and rapidly destabilizes pure retinol molecules, rendering both compounds chemically inert.",
        "solution": "Apply Benzoyl Peroxide in AM and Retinol in PM."
    },
    {
        "ingredient_a": "ascorbic acid",
        "ingredient_b": "copper peptide",
        "severity": "High",
        "conflict_type": "Active Deactivation",
        "reason": "Direct L-ascorbic acid oxidizes copper ions, neutralizing antioxidant activity and degrading peptides.",
        "solution": "Use Vitamin C (Ascorbic Acid) in AM and Copper Peptides in PM."
    },
    {
        "ingredient_a": "ascorbic acid",
        "ingredient_b": "retinol",
        "severity": "Medium",
        "conflict_type": "pH / Irritation Conflict",
        "reason": "L-ascorbic acid requires a low pH (<3.5) while retinol prefers neutral pH (~5.5-6.0), causing potential stinging and reduced efficacy.",
        "solution": "Apply Vitamin C in the morning under SPF; use Retinol in the evening."
    },
    {
        "ingredient_a": "glycolic acid",
        "ingredient_b": "salicylic acid",
        "severity": "Medium",
        "conflict_type": "Over-Exfoliation Risk",
        "reason": "Layering direct AHAs over BHAs causes compounded stratum corneum thinning and skin sensitization.",
        "solution": "Alternate days of usage; avoid concurrent layering in the same routine."
    },
    {
        "ingredient_a": "carbomer",
        "ingredient_b": "dimethicone",
        "severity": "Medium",
        "conflict_type": "Physical Pilling Risk",
        "reason": "High-molecular weight carbomer polymers destabilize and ball up when layered directly over volatile silicone matrices under mechanical friction.",
        "solution": "Apply lightweight water-based carbomer gels first; allow 3-5 minutes absorption before silicone layers."
    },
    {
        "ingredient_a": "ascorbic acid",
        "ingredient_b": "glycolic acid",
        "severity": "Low",
        "conflict_type": "pH Stacking Irritation",
        "reason": "Stacking multiple low-pH acid formulations can compromise skin tolerance in sensitive barrier types.",
        "solution": "Separate by applying Vitamin C in AM and Glycolic Acid in PM."
    }
]

df_conflict_rules = pd.DataFrame(conflict_data)
df_conflict_rules.to_csv("data/conflict_rules.csv", index=False)
print(f"Saved conflict_rules.csv with {len(df_conflict_rules)} conflict rules.")

Saved conflict_rules.csv with 8 conflict rules.
